In [10]:
# control_with_inputs.py
import time
import csv
from pulsr2API import pulsr            # uses your uploaded pulsr2API.py (see filecite above)
import serial


In [17]:
# --- Configuration: change these to match your system ---
INPUT_PORT_1 = "COM12"     # Arduino input 1 (Windows) or "/dev/ttyUSB0" on Linux
INPUT_PORT_2 = "COM13"     # Arduino input 2
OUTPUT_PORT  = "COM14"     # Arduino that receives control (pulsr board)
BAUD_IN      = 9600       # typical input Arduinos baud
BAUD_OUT     = 2000000    # example used in your pulsr file; change if needed
SERIAL_TIMEOUT = 0.5      # seconds, ensures reads don't block forever
LOG_CSV = "input_log.csv"



In [18]:
def read_inputs_for_duration(ser1, ser2, duration, log_list):
    """
    Read from ser1 and ser2 for `duration` seconds, append (timestamp, p1, p2) tuples to log_list.
    If no data available from a port during a poll, keep last known or empty string.
    """
    end_t = time.time() + duration
    last1 = ""
    last2 = ""
    poll_interval = 0.05  # seconds between polls

    while time.time() < end_t:
        tstamp = time.time()
        try:
            if ser1.in_waiting > 0:
                raw1 = ser1.readline().decode('utf-8', errors='ignore').strip()
                if raw1 != "":
                    last1 = raw1
            # else keep last1

            if ser2.in_waiting > 0:
                raw2 = ser2.readline().decode('utf-8', errors='ignore').strip()
                if raw2 != "":
                    last2 = raw2
        except Exception as e:
            # non-fatal: record the exception text for debugging
            last1 = last1 if last1 else f"<err:{e}>"

        # record a sample (timestamp in ISO + fractional seconds)
        log_list.append((time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(tstamp)) + f".{int((tstamp%1)*1000):03d}", last1, last2))
        # also print to console for monitoring
        print(f"[{log_list[-1][0]}] IN1: {last1} | IN2: {last2}")
        time.sleep(poll_interval)



In [19]:
def main():
    ser_in1 = None
    ser_in2 = None
    pulsr_board = None
    logs = []

    try:
        # open input serials
        ser_in1 = serial.Serial(INPUT_PORT_1, BAUD_IN, timeout=SERIAL_TIMEOUT)
        ser_in2 = serial.Serial(INPUT_PORT_2, BAUD_IN, timeout=SERIAL_TIMEOUT)
        print("Input serial ports opened.")

        # initialize pulsr board (output/control)
        pulsr_board = pulsr()
        # set a small timeout on the pulsr serial so its read_until() won't block forever
        pulsr_board.pulsr2_coms.timeout = SERIAL_TIMEOUT
        pulsr_board.initialize_communication(BAUD_OUT, OUTPUT_PORT)
        # ensure the pulsr serial also uses the timeout after opening
        pulsr_board.pulsr2_coms.timeout = SERIAL_TIMEOUT
        print("Pulsr (output) initialized.")

        # Optional: small wait for Arduinos to settle
        time.sleep(1.0)

        # --- Sequence ---
        print("\n=== Sequence start: RIGHT 5s ===")
        pulsr_board.motion3(0)      # enable east/right motion but don't block
        read_inputs_for_duration(ser_in1, ser_in2, 5.0, logs)
        pulsr_board.disable_motions()
        print("Disabled motions. (end of first RIGHT)")

        print("\n=== REST 2s ===")
        read_inputs_for_duration(ser_in1, ser_in2, 2.0, logs)

        print("\n=== Sequence continue: RIGHT 3s ===")
        pulsr_board.motion3(0)      # enable right motion again
        read_inputs_for_duration(ser_in1, ser_in2, 3.0, logs)
        pulsr_board.disable_motions()
        print("Disabled motions. (end of second RIGHT)")

        print("\n=== Sequence complete: motor stopped ===")

        # Save log to CSV
        with open(LOG_CSV, "w", newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["timestamp", "input1_raw", "input2_raw"])
            writer.writerows(logs)
        print(f"Input log saved to {LOG_CSV}")

    except Exception as exc:
        print("Error during run:", exc)

    finally:
        try:
            if pulsr_board is not None:
                pulsr_board.disable_motions()   # ensure motors are off
                pulsr_board.close_communication()
        except Exception:
            pass
        if ser_in1 and ser_in1.is_open:
            ser_in1.close()
        if ser_in2 and ser_in2.is_open:
            ser_in2.close()
        print("Closed all serial connections. Exiting.")


In [20]:
main()

Input serial ports opened.
pulsr communication check succesful
Pulsr (output) initialized.

=== Sequence start: RIGHT 5s ===
b''
[2025-08-23 13:39:14.974] IN1:  | IN2: 
[2025-08-23 13:39:15.026] IN1:  | IN2: 
[2025-08-23 13:39:15.077] IN1:  | IN2: 
[2025-08-23 13:39:15.127] IN1:  | IN2: 
[2025-08-23 13:39:15.178] IN1:  | IN2: 
[2025-08-23 13:39:15.229] IN1:  | IN2: 
[2025-08-23 13:39:15.280] IN1:  | IN2: 
[2025-08-23 13:39:15.331] IN1:  | IN2: 
[2025-08-23 13:39:15.381] IN1:  | IN2: 
[2025-08-23 13:39:15.432] IN1:  | IN2: 
[2025-08-23 13:39:15.483] IN1:  | IN2: 
[2025-08-23 13:39:15.534] IN1:  | IN2: 
[2025-08-23 13:39:15.586] IN1:  | IN2: 
[2025-08-23 13:39:15.636] IN1:  | IN2: 
[2025-08-23 13:39:15.687] IN1:  | IN2: 
[2025-08-23 13:39:15.738] IN1:  | IN2: 
[2025-08-23 13:39:15.789] IN1:  | IN2: 
[2025-08-23 13:39:15.840] IN1:  | IN2: 
[2025-08-23 13:39:15.891] IN1:  | IN2: 
[2025-08-23 13:39:15.944] IN1:  | IN2: 
[2025-08-23 13:39:15.995] IN1:  | IN2: 
[2025-08-23 13:39:16.046] IN1: 

In [21]:
main()

Input serial ports opened.
pulsr communication check succesful
Pulsr (output) initialized.

=== Sequence start: RIGHT 5s ===
b''
[2025-08-23 13:41:09.518] IN1:  | IN2: 
[2025-08-23 13:41:09.569] IN1:  | IN2: 
[2025-08-23 13:41:09.619] IN1:  | IN2: 
[2025-08-23 13:41:09.669] IN1:  | IN2: 
[2025-08-23 13:41:09.720] IN1:  | IN2: 
[2025-08-23 13:41:09.770] IN1:  | IN2: 
[2025-08-23 13:41:09.821] IN1:  | IN2: 
[2025-08-23 13:41:09.871] IN1:  | IN2: 
[2025-08-23 13:41:09.921] IN1:  | IN2: 
[2025-08-23 13:41:09.972] IN1:  | IN2: 
[2025-08-23 13:41:10.023] IN1:  | IN2: 
[2025-08-23 13:41:10.073] IN1:  | IN2: 
[2025-08-23 13:41:10.124] IN1:  | IN2: 
[2025-08-23 13:41:10.175] IN1:  | IN2: 
[2025-08-23 13:41:10.225] IN1:  | IN2: 
[2025-08-23 13:41:10.276] IN1:  | IN2: 
[2025-08-23 13:41:10.327] IN1:  | IN2: 
[2025-08-23 13:41:10.377] IN1:  | IN2: 
[2025-08-23 13:41:10.427] IN1:  | IN2: 
[2025-08-23 13:41:10.478] IN1:  | IN2: 
[2025-08-23 13:41:10.528] IN1:  | IN2: 
[2025-08-23 13:41:10.579] IN1: 